# Notebook 6: RAG Evaluation

**Goal**: Systematically measure RAG quality using automated metrics.

## Metrics covered:
- **Faithfulness** — Are all claims supported by retrieved context?
- **Answer Relevancy** — Does the answer address the question?
- **Context Precision** — What fraction of retrieved chunks is relevant?
- **Context Recall (heuristic)** — Is relevant info present in context?
- **ROUGE-L** — String overlap with reference answers
- **Latency** — p50/p90/p99 distribution


In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import os

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from evaluation.evaluator import (
    RAGEvaluator, EvaluationSample, EvaluationResult, EvaluationReport, StringMetrics
)

print('Evaluation framework loaded')

## 6.1 Build and Index Pipeline

In [ ]:
from rag_pipeline import RAGPipeline
from embeddings.embedding import EmbeddingEngine
from vector_store.vector_store import VectorStoreFactory
from generation.llm_interface import LLMInterface
from retrieval.reranker import CrossEncoderReranker

USE_OPENAI = bool(os.getenv('OPENAI_API_KEY'))

embedder = EmbeddingEngine(
    provider='sentence_transformers',
    model_name='all-MiniLM-L6-v2',
    use_cache=True, cache_dir='../.embed_cache',
)

vector_store = VectorStoreFactory.create(
    backend='faiss', dimension=embedder.dimension,
    index_type='flat', metric='cosine',
)

if USE_OPENAI:
    llm = LLMInterface(provider='openai', model='gpt-4o-mini', temperature=0.0)
else:
    llm = LLMInterface(provider='ollama', model='llama3.1:8b', temperature=0.0)

pipeline = RAGPipeline(
    embedder=embedder, vector_store=vector_store, llm=llm,
    reranker=CrossEncoderReranker(),
)
pipeline.index(['../sample_data/'])
print(f'Pipeline ready. Index size: {pipeline.index_size}')

## 6.2 Define Test Dataset

In [ ]:
# Ground-truth Q&A pairs for evaluation
TEST_SET = [
    {
        "question": "How many days do I have to request a refund?",
        "ground_truth": "Customers are eligible for a full refund within 30 days of purchase."
    },
    {
        "question": "How long does it take to process a refund?",
        "ground_truth": "Refunds are processed within 5-7 business days."
    },
    {
        "question": "What payment methods are accepted?",
        "ground_truth": "Visa, Mastercard, American Express, PayPal, and Apple Pay are accepted."
    },
    {
        "question": "How long does international shipping take?",
        "ground_truth": "International orders typically arrive in 10-15 business days."
    },
    {
        "question": "What is the price of the SmartWatch Pro?",
        "ground_truth": "The SmartWatch Pro costs $299."
    },
    {
        "question": "Can I return a digital download?",
        "ground_truth": "Digital downloads are non-refundable once accessed."
    },
    {
        "question": "How do I track my order?",
        "ground_truth": "You will receive a tracking email within 24 hours after shipping. You can also track under My Orders."
    },
    {
        "question": "What is the capital of Germany?",
        "ground_truth": "This information is not available in the documentation."
    },
]

print(f'Test set: {len(TEST_SET)} questions')

## 6.3 Run Pipeline and Collect Outputs

In [ ]:
# Run each test question through the pipeline
pipeline_outputs = []

for item in TEST_SET:
    q = item['question']
    r = pipeline.query(q)
    pipeline_outputs.append({
        'question': q,
        'answer': r.answer,
        'contexts': [chunk.content for chunk in r.retrieved_chunks],
        'ground_truth': item['ground_truth'],
        'latency_ms': r.total_latency_ms,
    })
    print(f'✓ Q: {q[:60]}')
    print(f'  A: {r.answer[:100]}...')
    print()

## 6.4 String-Based Metrics (No LLM)

In [ ]:
sm = StringMetrics()
string_results = []

for item in pipeline_outputs:
    rouge = sm.rouge_l(item['answer'], item['ground_truth'])
    token_f1 = sm.token_overlap_f1(item['answer'], item['ground_truth'])
    em = sm.exact_match(item['answer'], item['ground_truth'])
    ctx_recall = sm.context_recall_heuristic(item['answer'], item['contexts'])
    
    string_results.append({
        'question': item['question'][:40],
        'rouge_l': round(rouge, 4),
        'token_f1': round(token_f1, 4),
        'exact_match': em,
        'ctx_recall_heuristic': round(ctx_recall, 4),
        'latency_ms': round(item['latency_ms'], 1),
    })

df_str = pd.DataFrame(string_results)
print('String Metrics Results:')
print(df_str.to_string(index=False))

print(f'\nMean Scores:')
for col in ['rouge_l', 'token_f1', 'ctx_recall_heuristic']:
    print(f'  {col}: {df_str[col].mean():.4f}')

## 6.5 LLM-Judge Evaluation (Requires API Key)

In [ ]:
if USE_OPENAI:
    llm_fn = lambda p: pipeline._llm.complete(p).content
    
    evaluator = RAGEvaluator(
        llm_fn=llm_fn,
        metrics=['faithfulness', 'answer_relevancy', 'context_precision',
                 'context_recall_heuristic', 'rouge_l'],
    )

    samples = [
        EvaluationSample(
            question=item['question'],
            answer=item['answer'],
            contexts=item['contexts'],
            ground_truth=item['ground_truth'],
        )
        for item in pipeline_outputs
    ]

    report = evaluator.evaluate_dataset(samples)
    report.print_summary()
else:
    print('LLM-judge evaluation skipped (no OPENAI_API_KEY).')
    print('String metrics from 6.4 are still valid without an API key.')
    report = None

## 6.6 Latency Analysis

In [ ]:
latencies = [item['latency_ms'] for item in pipeline_outputs]

print('Latency Statistics (ms):')
print(f'  Mean:  {np.mean(latencies):.1f}')
print(f'  Std:   {np.std(latencies):.1f}')
print(f'  Min:   {np.min(latencies):.1f}')
print(f'  p50:   {np.percentile(latencies, 50):.1f}')
print(f'  p90:   {np.percentile(latencies, 90):.1f}')
print(f'  p99:   {np.percentile(latencies, 99):.1f}')
print(f'  Max:   {np.max(latencies):.1f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

questions_short = [item['question'][:25] + '...' for item in pipeline_outputs]
axes[0].barh(questions_short, latencies, color='steelblue')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_title('Per-Query Latency')
axes[0].axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean={np.mean(latencies):.0f}ms')
axes[0].legend()

axes[1].hist(latencies, bins=8, edgecolor='black', color='steelblue', alpha=0.8)
axes[1].axvline(np.percentile(latencies, 50), color='orange', linestyle='--', label='p50')
axes[1].axvline(np.percentile(latencies, 90), color='red', linestyle='--', label='p90')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_title('Latency Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('eval_latency.png', dpi=100)
plt.show()

## 6.7 Retrieval Quality Analysis

In [ ]:
# Analyze how many chunks were retrieved and their score distribution
retrieval_analysis = []
for item in pipeline_outputs:
    r = pipeline.query(item['question'])
    scores = [chunk.score for chunk in r.retrieved_chunks]
    retrieval_analysis.append({
        'question': item['question'][:35],
        'n_chunks': len(r.retrieved_chunks),
        'top1_score': scores[0] if scores else 0,
        'mean_score': np.mean(scores) if scores else 0,
        'score_spread': max(scores) - min(scores) if len(scores) > 1 else 0,
    })

df_ret = pd.DataFrame(retrieval_analysis)
print('Retrieval Quality:')
print(df_ret.to_string(index=False))

## 6.8 Failure Mode Analysis

In [ ]:
# Identify questions with lowest ROUGE scores
df_str_sorted = df_str.sort_values('rouge_l')

print('Questions with LOWEST ROUGE-L scores (potential failures):')
print('='*70)
for _, row in df_str_sorted.head(3).iterrows():
    print(f'Q:       {row["question"]}')
    print(f'ROUGE-L: {row["rouge_l"]:.4f}')
    
    # Get the pipeline output
    for item in pipeline_outputs:
        if item['question'][:40] == row['question']:
            print(f'Answer:  {item["answer"][:150]}')
            print(f'GT:      {item["ground_truth"][:150]}')
            break
    print('-'*70)

print('\nCommon RAG failure modes:')
failure_modes = [
    ('Retrieval miss',     'Question matches no stored chunk → empty context → hallucination'),
    ('Context overflow',   'Too many chunks dilute the answer signal'),
    ('Keyword-semantic gap', 'BM25 fails on synonyms; dense fails on acronyms'),
    ('Multi-hop failure',  'Answer requires combining info from 2+ chunks'),
    ('Prompt injection',   'Malicious query overrides system prompt'),
    ('Stale knowledge',    'Index not refreshed; outdated chunks returned'),
    ('PII leakage',        'Sensitive data in chunks exposed in answers'),
]
for name, desc in failure_modes:
    print(f'  • {name:<25} — {desc}')

## 6.9 Metrics Radar Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use mean scores from string metrics as proxy
metrics = ['ROUGE-L', 'Token F1', 'Ctx Recall', 'Latency Score']
values = [
    df_str['rouge_l'].mean(),
    df_str['token_f1'].mean(),
    df_str['ctx_recall_heuristic'].mean(),
    max(0, 1 - np.mean(latencies) / 2000),   # normalized latency score
]

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
values_plot = values + [values[0]]
angles_plot = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles_plot, values_plot, alpha=0.25, color='steelblue')
ax.plot(angles_plot, values_plot, 'o-', color='steelblue', linewidth=2)
ax.set_xticks(angles)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('RAG System Quality Radar', fontsize=14, pad=20)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)

plt.tight_layout()
plt.savefig('eval_radar.png', dpi=100)
plt.show()

print('Mean scores:')
for m, v in zip(metrics, values):
    print(f'  {m:<20}: {v:.4f}')

## 6.10 Save Evaluation Report

In [ ]:
import json

report_data = {
    'aggregate': {
        'rouge_l': df_str['rouge_l'].mean(),
        'token_f1': df_str['token_f1'].mean(),
        'ctx_recall_heuristic': df_str['ctx_recall_heuristic'].mean(),
    },
    'latency': {
        'p50': float(np.percentile(latencies, 50)),
        'p90': float(np.percentile(latencies, 90)),
        'mean': float(np.mean(latencies)),
    },
    'samples': pipeline_outputs,
}

with open('../evaluation_report.json', 'w') as f:
    json.dump(report_data, f, indent=2)
print('Report saved to ../evaluation_report.json')

print('\n=== FINAL EVALUATION SUMMARY ===')
print(f'Test questions:   {len(TEST_SET)}')
print(f'Mean ROUGE-L:     {df_str["rouge_l"].mean():.4f}')
print(f'Mean Token F1:    {df_str["token_f1"].mean():.4f}')
print(f'Ctx Recall (heur):{df_str["ctx_recall_heuristic"].mean():.4f}')
print(f'Mean Latency:     {np.mean(latencies):.0f} ms')
print(f'p90 Latency:      {np.percentile(latencies, 90):.0f} ms')

## Evaluation Benchmark Reference

| Metric | Excellent | Good | Needs Improvement |
|---|---|---|---|
| Faithfulness | > 0.90 | 0.75–0.90 | < 0.75 |
| Answer Relevancy | > 0.85 | 0.70–0.85 | < 0.70 |
| Context Precision | > 0.80 | 0.65–0.80 | < 0.65 |
| ROUGE-L | > 0.50 | 0.30–0.50 | < 0.30 |
| p90 Latency | < 500ms | 500ms–2s | > 2s |

**Improvement levers**:
- Low faithfulness → Stricter system prompt + better chunking
- Low context precision → Cross-encoder reranking
- Low ROUGE-L → Better retrieval or larger top-k
- High latency → HNSW index + embedding cache + smaller reranker